In [0]:
from pyspark.sql.functions import col, when, expr

df = spark.read.table("mining.bronze.haul_events")
df = df.dropDuplicates(["cycle_id"])

# tonnage came in as a string. Make a clean numeric version 
df = df.withColumn("tonnage_num", expr("try_cast(reported_tonnage as double)"))

In [0]:
is_777_valid = (col("model") == "777") & (col("tonnage_num") >= 15) & (col("tonnage_num") <= 130)

is_785_valid = (col("model") == "785") & (col("tonnage_num") >= 25) & (col("tonnage_num") <= 195)

fields_present = col("cycle_id").isNotNull() & col("truck_id").isNotNull() & col("tonnage_num").isNotNull()

is_valid = (is_777_valid | is_785_valid) & fields_present

valid_df = df.filter(is_valid)
quarantine_df = df.filter(~is_valid)




In [0]:
print(valid_df.count(), quarantine_df.count())

In [0]:
valid_df = valid_df.withColumn(
    "quality_flag",
    when(
        ((col("model") == "777") & (col("tonnage_num") > 110)) | ((col("model") == "785") & (col("tonnage_num") > 165)), "overload_suspected"
    ).otherwise("ok")
)

valid_df.groupBy("quality_flag").count().show()

In [0]:
valid_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("mining.silver.haul_events")
quarantine_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("mining.silver.quarantine")

print(spark.read.table("mining.silver.haul_events").count())
print(spark.read.table("mining.silver.quarantine").count())

In [0]:
spark.read.table("mining.silver.haul_events").count()


In [0]:
spark.read.table("mining.silver.quarantine").count()

In [0]:
spark.read.json("/Volumes/mining/landing/telemetry/") \
    .selectExpr("max(reported_tonnage) as max_reported").show()

In [0]:
spark.read.table("mining.silver.quarantine").select("cycle_id", "model", "reported_tonnage").show()